# TMEsim — M0 and M1

Reference: Frangieh et al., *Multimodal pooled Perturb-CITE-seq screens in patient models define mechanisms of cancer immune evasion*, Nature Genetics (2021). [Paper](https://doi.org/10.1038/s41588-021-00779-1).

| Symbol / code | Description | Units |
|---|---|---|
| `t` | Time since TIL addition | h |
| `S`, `R`, `T` | Non-targeting, knockout, total tumor populations | normalized population units |
| `T0`, `S0`, `R0` | Initial populations | same as above |
| `E` | Constant TIL abundance, E = rho × T0 | cells in physical units |
| `rho` | Initial TIL:tumor ratio | - |
| `r` | Net tumor growth rate (division minus background death) | h⁻¹ |
| `k` | Per-T-cell killing coefficient | cell⁻¹ h⁻¹ |
| `kappa` | k × T0, normalized killing coefficient | h⁻¹ |
| `s_g` | Relative change in killing susceptibility for knockout g | - |
| `f` | Survival relative to matched no-TIL control | - |
| `e_g` | KO-to-control odds enrichment relative to no-TIL | - |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#Extended Data Fig. 3a: mean killing percentages at 48 h
#The three entries correspond to TIL:tumor ratios 1:1, 2:1 and 4:1
rho_data = np.array([1., 2., 4.]) # estimating the killing rate
#Convert percent killed to the fraction surviving relative to no TILs
f_data = 1 - np.array([30.96, 58.35, 75.0]) / 100
t_screen = 48

#Eq. 2: f = exp(-kappa * rho * t)
#Taking logs gives a straight line through zero with slope -kappa
#x is rho*t --> unweighted least-squares slope through zero
x = rho_data * t_screen
kappa = -np.sum(x * np.log(f_data)) / np.sum(x**2)

#Fig. 1b, melanoma line 2686, no TILs (rho = 0)
#The time-zero value is for a normalization reference, it's not an actual replicate
t_growth = np.array([0., 24., 48., 72.])
T_relative = np.array([1., 1.05, 1.40, 2.40])

#With no killing, T/T0 = exp(r*t), so log(T/T0) = r*t
#Estimated r using the zero-intercept least-squares approach from the figure
r = np.sum(t_growth * np.log(T_relative)) / np.sum(t_growth**2)
print("r =", r)
print("kappa =", kappa)

### Data and calibration assumptions

In [ ]:
def M0(t, r, kappa, rho, T0=1.0):
    #Accept a sequence of times in hours for simulation
    t = np.asarray(t)

    #Eq. 1: dT/dt = (r - kappa*rho)*T
    #Growth adds cells and constant TIL pressure removes them
    #The exponential is the exact solution
    T = T0 * np.exp((r - kappa * rho) * t)

    #Eq. 2: divide T by the matched no-TIL trajectory T0*exp(r*t)
    #Both initial abundance and shared growth cancel
    f = np.exp(-kappa * rho * t)
    return pd.DataFrame({'t': t, 'T': T, 'f': f})


def M1(t, r, kappa, rho, s_g, T0=1.0, R0=0.5):
    #s_g = -1 means no modeled killing of KO cells; lower values would imply negative killing. Both populations must start present.
    if s_g < -1 or not 0 < R0 < T0:
        raise ValueError('Require s_g >= -1 and 0 < R0 < T0.')
    t = np.asarray(t)
    S0 = T0 - R0

    #Eq. 3: non-targeting cells have the baseline susceptibility
    S = S0 * np.exp((r - kappa * rho) * t)

    #Eq. 4: KO cells share r but their killing rate is scaled by 1+s_g
    #Negative s_g means resistance; positive s_g means sensitization
    R = R0 * np.exp((r - kappa * rho * (1 + s_g)) * t)
    T = S + R

    #Total mixed-population survival relative to the same mixture without TILs
    #Unlike enrichment, this depends on the initial mixture
    f = T / (T0 * np.exp(r * t))

    #Eq. 5: (R/S) with TILs divided by (R/S) without TILs
    e_g = np.exp(-s_g * kappa * rho * t)

    #One row per time point, with column names matching the model symbols
    return pd.DataFrame({'t': t, 'S': S, 'R': R, 'T': T, 'f': f, 'e_g': e_g})

In [ ]:
#Sample the exact solutions every hour from 0 through 72 h
t = np.arange(73)
rho = 2  #Perturb-CITE-seq co-culture ratio in the model
s_g = 0  #Neutral reference until a gene-specific value is supplied for gene knockout ablations

baseline = M0(t, r, kappa, rho)
mixed = M1(t, r, kappa, rho, s_g)

#Show predicted total population for no-TIL and screen ratios
for dose in [0, 1, 2, 4]:
    result = M0(t, r, kappa, dose)
    plt.plot(result.t, result["T"], label=f'{dose}:1')
plt.xlabel('Time (h)')
plt.ylabel('T / T0')
plt.legend(title='TIL:tumor')
plt.show()

In [ ]:
#SAMPLE ABLATIONS FOR VERIFICATION AND DEBUGGING MODEL

#Removing net growth changes absolute populations but not normalized f
no_growth = M0(t, 0, kappa, rho)

#Remove killing while retaining the specified TIL ratio
no_killing = M0(t, r, 0, rho)

#Remove TILs instead; equivalent to no killing under this baseline
no_TIL = M0(t, r, kappa, 0)

#Equalize KO and non-targeting susceptibility. This is identical to
#the current neutral M1 example until a nonzero s_g is supplied
no_KO_effect = M1(t, r, kappa, rho, 0)